# MyAI K2 — 20M smoke on real 10M data (GPU)
Tokenizer-32k comes from K1 output (attached input). Re-streams 10M (~15 min), packs 512, trains MyAI-20M x500 AMP. Validates the loss trajectory on the real distribution before the 100M run.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1
!cp /kaggle/input/myai-data-10m/tokenizer-32k.json data/tokenizer_corpus/ && python -c "from myai.tokenizer.bpe import BPETokenizer; print('tok:', len(BPETokenizer.load('data/tokenizer_corpus/tokenizer-32k.json')))"

In [ ]:
!python scripts/stream_fineweb.py --max-tokens 10000000 --out data/raw/fineweb-10M.txt
!python scripts/prepare_tokenizer_corpus.py

In [ ]:
%%writefile configs/data-pack-fw10m.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fw10m-32k
seq_len: 512
val_ratio: 0.01

In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-fw10m.yaml --rows-per-shard 2048

In [ ]:
!python scripts/train.py --model-config configs/myai-20m.yaml --data-dir data/processed/fw10m-32k --max-steps 500 --batch-size 4 --accum 8 --amp --eval-every 100 --ckpt-every 250 --out-dir /kaggle/working/ckpts-20m --report /kaggle/working/train-20m.json

In [ ]:
!python scripts/eval_model.py --checkpoint /kaggle/working/ckpts-20m/final.pt --data-dir data/processed/fw10m-32k --report /kaggle/working/eval-20m.json
!ls -la /kaggle/working/ckpts-20m/ && cat /kaggle/working/train-20m.json